In [2]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
from matplotlib.colors import Normalize

# ===== 1. 读取数据 =====
file_path = '/Users/weiyingwan/Desktop/校内课程/大三下/数值天气预报/project/data/2026JanFeb.nc'
ds = xr.open_dataset(file_path)

# 提取500hPa位势高度并转换为位势米
z_500 = ds['z'].sel(pressure_level=500.0) / 9.80665
lat = ds.latitude.values
lon = ds.longitude.values

# 原始时间
times = ds.valid_time.values

# -----------------------
# 只保留每天00 UTC
# -----------------------
selected_indices = []

for i, t in enumerate(times):
    # 转成字符串，例如：
    # 2025-01-01T00:00:00
    time_str = np.datetime_as_string(t, unit='m')

    # 判断是否00:00
    if time_str.endswith("00:00"):
        selected_indices.append(i)

# 筛选数据
z_500 = z_500.isel(valid_time=selected_indices)
times = times[selected_indices]

print(f"筛选后共有 {len(times)} 天（每天00 UTC）")

print(f"数据信息:")
print(f"  时间数量: {len(times)}")
print(f"  500hPa位势高度场形状: {z_500.shape}")

# ===== 2. 设置总图参数 =====
n_rows, n_cols = 8, 8  # 8x8布局，共64个子图
n_subplots = n_rows * n_cols

# 检查时间数量
total_times = len(times)
n_pages = int(np.ceil(total_times / n_subplots))

print(f"总共有 {total_times} 个时次")
print(f"将生成 {n_pages} 张8×8大图")


# ==========================
# 3~6. 按64张一页生成大图
# ==========================

proj = ccrs.PlateCarree()

z_min, z_max = 5000, 5900
levels_fill = np.arange(z_min, z_max, 20)
levels_contour = np.arange(z_min, z_max, 40)

output_dir = '/Users/weiyingwan/Desktop/校内课程/大三下/数值天气预报/project/figures/case selection/case selection3'
os.makedirs(output_dir, exist_ok=True)

for page in range(n_pages):

    start_idx = page * n_subplots
    end_idx = min((page + 1) * n_subplots, total_times)

    print(f"\n正在绘制第 {page+1} 张大图：")
    print(f"{start_idx} -> {end_idx-1}")

    fig = plt.figure(figsize=(30,30))

    cf = None

    for local_i, global_i in enumerate(range(start_idx, end_idx)):

        ax = fig.add_subplot(
            n_rows,
            n_cols,
            local_i+1,
            projection=proj
        )

        time = times[global_i]

        z_now = z_500.isel(valid_time=global_i)

        ax.add_feature(
            cfeature.LAND,
            facecolor='lightgray',
            alpha=0.3,
            linewidth=0.3
        )

        ax.add_feature(
            cfeature.COASTLINE,
            linewidth=0.5
        )

        ax.add_feature(
            cfeature.BORDERS,
            linewidth=0.3,
            linestyle='--',
            alpha=0.5
        )

        cf = ax.contourf(
            lon,
            lat,
            z_now,
            levels=levels_fill,
            cmap='RdBu_r',
            extend='both'
        )

        contour = ax.contour(
            lon,
            lat,
            z_now,
            levels=levels_contour,
            colors='black',
            linewidths=0.8
        )

        ax.clabel(
            contour,
            inline=True,
            fontsize=6,
            fmt='%d'
        )

        ax.set_extent(
            [70,150,20,60],
            crs=ccrs.PlateCarree()
        )

        if local_i % n_cols == 0:

            gl = ax.gridlines(
                draw_labels=True,
                linewidth=0.3,
                color='gray',
                alpha=0.5,
                linestyle='--',
                dms=True
            )

            gl.right_labels=False
            gl.top_labels=False

        elif local_i < n_cols:

            gl = ax.gridlines(
                draw_labels=True,
                linewidth=0.3,
                color='gray',
                alpha=0.5,
                linestyle='--',
                dms=True
            )

            gl.right_labels=False
            gl.bottom_labels=False

        else:

            ax.gridlines(
                draw_labels=False,
                linewidth=0.3,
                color='gray',
                alpha=0.5,
                linestyle='--'
            )

        time_str = np.datetime_as_string(
            time,
            unit='D'
        )

        ax.set_title(
            time_str,
            fontsize=10,
            fontweight='bold',
            pad=3
        )

    fig.suptitle(
        f'ERA5 500 hPa Geopotential Height\nPage {page+1}/{n_pages}',
        fontsize=20,
        fontweight='bold',
        y=0.98
    )

    cbar_ax = fig.add_axes(
        [0.92,0.25,0.02,0.5]
    )

    cbar = fig.colorbar(
        cf,
        cax=cbar_ax,
        orientation='vertical'
    )

    cbar.set_label(
        'Geopotential Height (gpm)',
        fontsize=16
    )

    cbar.ax.tick_params(
        labelsize=12
    )

    plt.subplots_adjust(
        left=0.05,
        right=0.9,
        top=0.95,
        bottom=0.05,
        wspace=0.1,
        hspace=0.15
    )

    png_path = os.path.join(
        output_dir,
        f"z500_composite_8x8_page{page+1}.png"
    )

    pdf_path = os.path.join(
        output_dir,
        f"z500_composite_8x8_page{page+1}.pdf"
    )

    plt.savefig(
        png_path,
        dpi=200,
        bbox_inches='tight'
    )

    plt.savefig(
        pdf_path,
        bbox_inches='tight'
    )

    plt.close()

    print(f"保存完成：{png_path}")

print("\n全部完成！")


筛选后共有 59 天（每天00 UTC）
数据信息:
  时间数量: 59
  500hPa位势高度场形状: (59, 721, 1440)
总共有 59 个时次
将生成 1 张8×8大图

正在绘制第 1 张大图：
0 -> 58
保存完成：/Users/weiyingwan/Desktop/校内课程/大三下/数值天气预报/project/figures/case selection/case selection3/z500_composite_8x8_page1.png

全部完成！
